In [1]:
%pip install -q pandas numpy sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

from dotenv import load_dotenv
import os

In [3]:
data = pd.read_csv('./WA_Fn−UseC_−Telco−Customer−Churn.csv')
data.shape

(7043, 21)

In [4]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [6]:
data['customerID'].nunique()

7043

Смотрим на уникальные значения по столбцам:

In [7]:
for col in data.columns:
    print(f'{col}\n{data[col].unique()}')

customerID
['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
gender
['Female' 'Male']
SeniorCitizen
[0 1]
Partner
['Yes' 'No']
Dependents
['No' 'Yes']
tenure
[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
PhoneService
['No' 'Yes']
MultipleLines
['No phone service' 'No' 'Yes']
InternetService
['DSL' 'Fiber optic' 'No']
OnlineSecurity
['No' 'Yes' 'No internet service']
OnlineBackup
['Yes' 'No' 'No internet service']
DeviceProtection
['No' 'Yes' 'No internet service']
TechSupport
['No' 'Yes' 'No internet service']
StreamingTV
['No' 'Yes' 'No internet service']
StreamingMovies
['No' 'Yes' 'No internet service']
Contract
['Month-to-month' 'One year' 'Two year']
PaperlessBilling
['Yes' 'No']
PaymentMethod
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (a

In [8]:
min(data['MonthlyCharges']), max(data['MonthlyCharges'])

(18.25, 118.75)

In [9]:
# индексы строк, в которых стоит ' ' вместо числа в TotalCharges
idx = data[data['TotalCharges'] == ' '].index
idx

Index([488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754], dtype='int64')

In [10]:
data.iloc[idx][['tenure', 'TotalCharges']]

,tenure,TotalCharges
488,0,
753,0,
936,0,
1082,0,
1340,0,
3331,0,
3826,0,
4380,0,
5218,0,
6670,0,


Проверяем, что tenure = 0 <=> когда TotalCharges содержит ' ':

In [11]:
data[data['tenure'] == 0].index == idx

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True])

In [12]:
data['TotalCharges'] = data['TotalCharges'].replace(' ', np.nan).astype(float)
min(data['TotalCharges']), max(data['TotalCharges'])

(18.8, 8684.8)

### Преобразование колонок и конвертация в .sql

Сначала создадим копию изначальных данных:

In [13]:
customers = data.copy()
customers.shape

(7043, 21)

Введем словари для преобразования данных:

In [14]:
contract_types = {'Month-to-month': 1, 'One year': 2, 'Two year': 3}
payment_methods = {'Electronic check': 1, 'Mailed check': 2,
                   'Bank transfer (automatic)': 3, 'Credit card (automatic)': 4}
internet_services = {'DSL': 1, 'Fiber optic': 2, 'No': 3}

Преобразуем булевые признаки:

In [15]:
bool_cols = ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in bool_cols:
    if col == 'SeniorCitizen':
        customers[col] = customers[col].map({1: True, 0: False})
    else:
        customers[col] = customers[col].map({'Yes': True, 'No': False})

In [16]:
customers['TotalCharges'] = customers['TotalCharges'].replace(' ', np.nan).astype(float)
customers['MultipleLines'] = customers['MultipleLines'].map({'Yes': True, 'No': False, 'No phone service': np.nan})

In [17]:
customers['internet_service_id'] = [internet_services[val] for val in customers['InternetService']]
customers['contract_type_id'] = [contract_types[val] for val in customers['Contract']]
customers['payment_method_id'] = [payment_methods[val] for val in customers['PaymentMethod']]

# убирааем лишние колонки
customers = customers.drop(columns=['InternetService', 'Contract', 'PaymentMethod'])

In [18]:
options = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for option in options:
    customers[option] = customers[option].map({'Yes': True, 'No': False, 'No internet service': np.nan})

Переименовываем колонки для единообразия как в DDL:

In [19]:
customers = customers.rename(columns={'customerID': 'customer_id',
                                      'SeniorCitizen': 'is_senior',
                                      'Partner': 'has_partner',
                                      'Dependents': 'has_dependents',
                                      'tenure': 'tenure_months',
                                      'PhoneService': 'has_phone_service',
                                      'MultipleLines': 'has_multiple_lines',
                                      'OnlineSecurity': 'has_online_security',
                                      'OnlineBackup' : 'has_online_backup',
                                      'DeviceProtection': 'has_device_protection',
                                      'TechSupport': 'has_tech_support',
                                      'StreamingTV': 'has_streaming_tv',
                                      'StreamingMovies': 'has_streaming_movies',
                                      'PaperlessBilling': 'has_paperless_billing',
                                      'MonthlyCharges': 'monthly_charges',
                                      'TotalCharges': 'total_charges',
                                      'Churn': 'has_churned'})

Меняем порядок колонок, чтобы он совпадал с порядком в скрипте DDL:

In [20]:
customers = customers.reindex(columns=['customer_id', 'gender', 'is_senior', 'has_partner', 'has_dependents',
                                       'tenure_months', 'monthly_charges', 'total_charges', 'has_phone_service',
                                       'has_multiple_lines', 'internet_service_id', 'has_online_security',
                                       'has_online_backup', 'has_device_protection', 'has_tech_support',
                                       'has_streaming_tv', 'has_streaming_movies', 'contract_type_id',
                                       'has_paperless_billing', 'payment_method_id', 'has_churned'])

In [21]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            7043 non-null   object 
 1   gender                 7043 non-null   object 
 2   is_senior              7043 non-null   bool   
 3   has_partner            7043 non-null   bool   
 4   has_dependents         7043 non-null   bool   
 5   tenure_months          7043 non-null   int64  
 6   monthly_charges        7043 non-null   float64
 7   total_charges          7032 non-null   float64
 8   has_phone_service      7043 non-null   bool   
 9   has_multiple_lines     6361 non-null   object 
 10  internet_service_id    7043 non-null   int64  
 11  has_online_security    5517 non-null   object 
 12  has_online_backup      5517 non-null   object 
 13  has_device_protection  5517 non-null   object 
 14  has_tech_support       5517 non-null   object 
 15  has_

In [22]:
customers.to_csv('customers_transformed.csv', index=False)

Теперь загружаем customers в PostgreSQL:

In [23]:
from sqlalchemy import create_engine, URL

In [24]:
load_dotenv('.env')

engine = create_engine(URL.create('postgresql+psycopg2',
                                  username=os.getenv('PG_USER'),
                                  password=os.getenv('PG_PASSWORD'),
                                  host=os.getenv('PG_HOST', 'localhost'),
                                  port=int(os.getenv('PG_PORT', '5432')),
                                  database=os.getenv('PG_DBNAME', 'postgres'),
))

In [25]:
customers.to_sql('customers', engine, schema='telco_churn', if_exists='append', index=False)

43

In [26]:
res = pd.read_sql('SELECT COUNT(*) AS customers_cnt FROM telco_churn.customers', engine)
print(res.to_string(index=False))

 customers_cnt
          7043
